# Benchmark with the CRECK mechanism

The feedstock is characterised from its ultimate analysis by reference mixtures (Ranzi et al., 2017) and the 32 reactions of Debiagi et al. (2018) are integrated at 30 °C/min (Section 4.1). The products are mapped onto the lumps of the framework: every volatile species other than water and the four measured gases is counted as tar.

In [ ]:
import os
import shutil
import sys
import tempfile
from pathlib import Path
ARTICLE = Path.cwd().parent / "results"
OUTPUT = Path.cwd() / "output" / "05_creck_benchmark"
OUTPUT.mkdir(parents=True, exist_ok=True)
if (ARTICLE / "best_vectors.npz").exists():
    handle, temporary = tempfile.mkstemp(dir=OUTPUT, suffix=".npz")
    os.close(handle)
    shutil.copyfile(ARTICLE / "best_vectors.npz", temporary)
    os.replace(temporary, OUTPUT / "best_vectors.npz")
os.environ.setdefault("CCPYRO_RESULTS", str(OUTPUT))
os.environ.setdefault("CCPYRO_FIGDATA", str(OUTPUT / "figdata"))
try:
    import ccpyro  # installed with `pip install -e .`
except ImportError:  # running from a clone without installation
    sys.path.insert(0, str(Path.cwd().parent / "src"))
    import ccpyro
import numpy as np
import pandas as pd
np.set_printoptions(precision=6, suppress=True)
print("ccpyro", ccpyro.__version__)

In [ ]:
from ccpyro import creck as C
mech = C.load_mechanism()
print(len(mech.reactions), "reactions; largest elemental imbalance:",
      max(float(np.abs(r.imbalance()).max()) for r in mech.reactions))
nodes, curves, characterisation = C.run_benchmark()
for name, ch in characterisation.items():
    split = {k: round(v, 3) for k, v in ch["splitting"].items()}
    print(f"{name:10s} step {ch['step']}, exact {ch['exact']}, splitting {split}")

In [ ]:
nodes[["feedstock", "T", "w_char", "w_tar", "x_t", "y_t", "w_water", "w_gas", "phi_CH4", "closure"]].round(4)

In [ ]:
published = ARTICLE / "product_yields.csv"
if published.exists():
    merged = C.compare_with_framework(nodes, pd.read_csv(published))
    show = ["feedstock", "T", "w_char_creck", "w_char_est", "w_tar_creck", "w_tar_est", "y_t_creck", "y_t_est",
            "phi_CH4", "phi_CH4_exp"]
    display(merged[show].round(3))